In [ ]:
# ============================================================
# STEP 0: Imports
# ============================================================
import torch
import torchvision.transforms as transforms
from torchvision import datasets, models
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from umap import UMAP
from sklearn.preprocessing import StandardScaler

# ============================================================
# STEP 1: Load dataset
# ============================================================
# Example: Assume your dataset is in ./data/train/<class_name>/
# Replace with your actual dataset path
#data_dir = "C:/Users/ekadw/Documents/GitHub/Tomato_Disease_Detector_Deployed/examples"

data_dir = "C:/Users/ekadw/Documents/DATA/Leaf_Disease/Tomato20000/full"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

dataset = datasets.ImageFolder(root=data_dir, transform=transform)
loader = DataLoader(dataset, batch_size=32, shuffle=False, num_workers=2)

class_names = dataset.classes
print(f"Found {len(dataset)} images across {len(class_names)} classes: {class_names}")

# ============================================================
# STEP 2: Extract features using pretrained CNN
# ============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"

# Use pretrained ResNet50 backbone
model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
model = torch.nn.Sequential(*(list(model.children())[:-1]))  # remove classifier
model.eval().to(device)

features = []
labels = []

with torch.no_grad():
    for imgs, lbls in loader:
        imgs = imgs.to(device)
        output = model(imgs)              # shape: (batch, 2048, 1, 1)
        output = output.squeeze()         # shape: (batch, 2048)
        features.append(output.cpu().numpy())
        labels.extend(lbls.numpy())

features = np.vstack(features)
labels = np.array(labels)

print(f"Extracted feature matrix shape: {features.shape}")

# ============================================================
# STEP 3: Normalize + Reduce with UMAP
# ============================================================
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

umap = UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
embeddings_2d = umap.fit_transform(features_scaled)

# ============================================================
# STEP 4: Plot clusters
# ============================================================
plt.figure(figsize=(10, 8))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1],
                      c=labels, cmap='tab10', s=10, alpha=0.8)
plt.legend(handles=scatter.legend_elements()[0],
           labels=class_names, title="Classes",
           loc="best", fontsize=8)
plt.title("UMAP Projection of CNN Feature Space")
plt.xlabel("UMAP Dimension 1")
plt.ylabel("UMAP Dimension 2")
plt.tight_layout()
plt.show()


Found 28327 images across 11 classes: ['Bacterial_spot', 'Early_blight', 'Late_blight', 'Leaf_Mold', 'Septoria_leaf_spot', 'Spider_mites Two-spotted_spider_mite', 'Target_Spot', 'Tomato_Yellow_Leaf_Curl_Virus', 'Tomato_mosaic_virus', 'healthy', 'powdery_mildew']
